# Report del progetto - Forecasting della produzione lorda di energia elettrica

## Obiettivo
L'obiettivo del progetto e costruire una pipeline completa di forecasting sulla serie storica della produzione lorda totale di energia elettrica ricavata da Tavola 1.14. Il flusso implementato copre quattro passaggi principali: analisi descrittiva, preprocessing, modellazione statistica e modellazione machine learning.

## Dataset e impostazione generale
La serie utilizzata e annuale, copre il periodo 1883-2014 e contiene 131 osservazioni valide. Il problema e reso difficile da tre aspetti principali: trend di lungo periodo molto marcato, forte eterogeneita di scala tra inizio e fine serie e presenza di variazioni locali non spiegabili solo con il trend globale.

## Struttura della pipeline
1. Step 1: analisi descrittiva della serie originale.
2. Step 2: preprocessing leakage-safe con split temporale, trasformazioni e test statistici.
3. Step 3: confronto tra modelli statistici SARIMA e Holt-Winters.
4. Step 4: confronto tra modelli di machine learning basati su lag.

## Nota metodologica
Il report descrive cio che viene effettivamente eseguito dalla pipeline corrente. La parte neurale e stata prevista nella progettazione del preprocessing, ma non rientra ancora nel flusso operativo eseguito da main.py.

## Step 1 - Analisi descrittiva

Nel primo step la pipeline carica la serie target dal file CSV in formato ISTAT, pulisce i valori testuali, ricostruisce la serie numerica annuale e produce sia tabelle statistiche sia grafici descrittivi.

### Attivita eseguite
1. Caricamento della serie di produzione lorda totale e conversione dei valori nel formato numerico corretto.
2. Calcolo della distribuzione di frequenza e delle misure di tendenza centrale.
3. Calcolo delle misure di dispersione, con particolare attenzione a deviazione standard, coefficiente di variazione e IQR.
4. Ricerca di outlier globali tramite regola IQR.
5. Validazione numerica del trend con regressione lineare e correlazione di Spearman.
6. Ricerca di anomalie locali sulle variazioni anno su anno tramite rolling median, rolling MAD e fallback con rolling standard deviation.
7. Generazione dei grafici descrittivi e salvataggio degli artifact di output.

### Evidenze principali
- Media della serie: circa 88683.
- Mediana: 20782.
- Deviazione standard: circa 107127.
- Coefficiente di variazione: circa 1.21, quindi elevata eterogeneita relativa.
- Outlier globali IQR: 0.
- Trend lineare molto forte: slope circa 2549 unita per anno.
- Significativita del trend molto elevata: p-value circa 1.27e-50.
- Capacita esplicativa del trend lineare gia alta: R^2 circa 0.825.
- Correlazione monotona con il tempo quasi perfetta: Spearman rho circa 0.998.
- Outlier locali sulle variazioni YoY: 2 casi su 130 variazioni, pari a circa 1.5%.

### Conclusioni dello step
La serie presenta un trend crescente strutturale e molto marcato, senza outlier globali sui livelli ma con poche anomalie locali sulle variazioni anno su anno. Questo implica che il problema principale non e la presenza di valori estremi isolati, ma la non stazionarieta dovuta al trend e al cambio di scala nel tempo. Di conseguenza, lo step successivo deve concentrarsi soprattutto su trasformazioni e differenziazione.

## Step 2 - Preprocessing operativo

Il preprocessing e stato progettato per essere leakage-safe e per separare in modo rigoroso il comportamento della serie nelle tre porzioni train, validation e test.

### Attivita eseguite
1. Validazione e ordinamento della serie in input.
2. Split temporale in train, validation e test senza shuffle.
3. Applicazione delle trasformazioni deterministiche configurate: log1p, eventuale power transform, differencing iterativo.
4. Eventuale scaling con fit sul solo train per evitare leakage.
5. Esecuzione dei test di stazionarieta ADF e KPSS e, quando richiesto, del test di normalita Shapiro-Wilk.
6. Calcolo degli outlier locali sulle variazioni YoY anche a supporto della diagnostica del preprocessing.
7. Produzione dei grafici di confronto raw vs transformed, split temporale, ACF/PACF e anomalie locali.

### Evidenze principali
- Split corrente: 90 osservazioni di train, 19 di validation e 20 di test.
- La trasformazione scelta nel run salvato e: log1p + differenziazione di ordine 2 + nessuno scaling.
- Sul train la serie preprocessata risulta stazionaria sia per ADF sia per KPSS.
- Anche il validation mantiene esito coerente con una serie stazionaria.
- Sul test il KPSS resta accettabile, mentre ADF non conferma pienamente la stazionarieta.

### Conclusioni dello step
Il preprocessing riesce a stabilizzare bene la serie nella parte usata per l'addestramento e per la validazione, che sono i segmenti decisivi per la scelta dei modelli. Rimane invece una minore stabilita sul test, segnale coerente con il fatto che la parte finale della serie conserva una dinamica difficile da catturare completamente. In altre parole, il preprocessing migliora il problema ma non elimina tutta la complessita predittiva della serie.

## Step 2b - Selezione automatica della configurazione di preprocessing

Dopo il preprocessing operativo, la pipeline valuta diverse configurazioni candidate e sceglie automaticamente quella piu adatta al tipo di modello da addestrare. Questo passaggio e gestito dal modulo di autoconfigurazione.

### Attivita eseguite
1. Definizione di insiemi di candidati distinti per modelli statistici, machine learning e reti neurali.
2. Valutazione di ogni candidato sul train tramite i test di stazionarieta.
3. Ranking del candidato migliore con criteri diversi a seconda del profilo di utilizzo.
4. Costruzione della configurazione finale e salvataggio su file JSON per rendere il flusso riproducibile.

### Evidenze principali
- Nel run storico disponibile, tra i candidati valutati l'unica configurazione chiaramente stazionaria sia per ADF sia per KPSS sul train e risultata log1p + diff2 + no scaling.
- Dopo il refactor, la pipeline non usa piu un solo criterio universale, ma tre profili distinti: statistical, ml e neural.
- I runner statistici sono ora collegati al profilo statistical, mentre quelli ML al profilo ml.

### Conclusioni dello step
La conclusione metodologica piu importante e che non esiste un preprocessing universalmente migliore per tutte le famiglie di modelli. Per questo la selezione automatica e stata separata in tre profili distinti. Il risultato e un design piu coerente: i modelli statistici privilegiano stazionarita e parsimonia, mentre i modelli ML e neural possono privilegiare trasformazioni e scaling piu adatti alla loro struttura.

## Step 3 - Modellazione statistica

Nel terzo step la pipeline confronta due famiglie di modelli statistici: SARIMA e Holt-Winters. Il confronto viene effettuato su validation e test, con selezione finale del winner sulla validation per evitare leakage metodologico.

### Attivita eseguite
1. Addestramento di una griglia di configurazioni SARIMA.
2. Addestramento del benchmark Holt-Winters.
3. Valutazione su validation e test.
4. Costruzione della tabella di summary, diagnostica dei residui e tabella forecast.
5. Selezione del modello vincitore con criterio corretto basato sulla validation.

### Evidenze principali
- Winner dello step statistico: SARIMA.
- Il best SARIMA selezionato nel run corrente corrisponde a una configurazione molto semplice, con ordine non stagionale minimale.
- RMSE originale su validation del winner SARIMA: circa 6934.
- MAE originale su validation del winner SARIMA: circa 5414.
- MAPE originale su validation del winner SARIMA: circa 2.97%.
- Holt-Winters resta competitivo, ma risulta peggiore del winner soprattutto sulla validation in scala originale.

### Conclusioni dello step
Il confronto statistico mostra che una soluzione semplice e regolarizzata puo essere piu robusta di configurazioni piu complesse. La conclusione raggiunta e che SARIMA costituisce il riferimento statistico principale della pipeline attuale. Inoltre, la correzione del criterio di selezione del winner ha eliminato il rischio di scegliere il modello usando informazioni del test set.

## Step 4 - Modellazione machine learning

Nel quarto step la pipeline costruisce modelli di regressione supervisionata su feature di lag e confronta diverse famiglie di modelli ad albero.

### Attivita eseguite
1. Generazione delle feature autoregressive basate sui lag della serie preprocessata.
2. Addestramento e confronto di decision tree, random forest e gradient boosting.
3. Selezione delle feature tramite importanza.
4. Valutazione finale su validation e test e salvataggio del modello vincitore.

### Evidenze principali
- Winner dello step ML: gradient boosting con lookback 6.
- Feature selezionate: lag_1, lag_5, lag_3, lag_2, lag_4, lag_6.
- RMSE originale su validation: circa 7709.
- MAE originale su validation: circa 5945.
- MAPE originale su validation: circa 3.06%.
- RMSE originale su test: circa 72837.
- MAE originale su test: circa 65172.
- MAPE originale su test: circa 22.06%.
- Il decision tree mostra un comportamento chiaramente overfit: errore nullo in validation ma degrado molto forte in test.

### Conclusioni dello step
All'interno della famiglia ML il gradient boosting e il modello piu solido, ma il salto di errore tra validation e test mostra che la generalizzazione fuori campione rimane difficile. La conclusione operativa e che il machine learning classico, pur competitivo, e molto piu esposto a overfitting rispetto a quanto suggeriscono le metriche di validation. Questo rende necessaria una lettura prudente dei risultati e conferma l'importanza di una validazione rigorosa.

## Conclusioni generali

L'analisi svolta porta a quattro conclusioni principali.

1. La serie ha un trend crescente molto forte e una scala che cambia drasticamente nel tempo; di conseguenza la stazionarizzazione non e opzionale, ma e un requisito del problema.
2. Il preprocessing migliora sensibilmente la trattabilita della serie, ma non elimina del tutto la difficolta predittiva sulla parte finale del campione.
3. Nel blocco statistico, SARIMA rappresenta il miglior compromesso attuale ed e il winner della pipeline.
4. Nel blocco machine learning, gradient boosting e il migliore tra i modelli provati, ma mostra un gap significativo tra validation e test, segnale di generalizzazione ancora limitata.

Nel complesso, la pipeline e oggi solida dal punto di vista metodologico: separa le fasi, salva gli artifact, usa validation per selezionare i modelli e mantiene il test come verifica finale. Il passo naturale successivo non e aggiungere complessita indiscriminata, ma migliorare la capacita di generalizzazione, in particolare sul blocco ML e sul futuro blocco neurale.